# Biomedical Data Analytics — Assignment 2
## Machine Learning Classification on Pima Indians Diabetes Dataset

| | |
|:---|:---|
| **Course** | Biomedical Data Analytics |
| **Institution** | Cairo University — Faculty of Engineering |
| **Student Name** | Mahmoud Mohamed Abdelfattah |
| **Student ID** | 4220142 |
| **Date** | 12 May 2026 |

---

## Dataset Overview

| Property | Details |
|:---|:---|
| **Name** | Pima Indians Diabetes Database |
| **Source** | [Kaggle — UCI Pima Indians Diabetes](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) |
| **Rows** | 768 |
| **Columns** | 9 (8 features + 1 target) |
| **Target Variable** | `Outcome` (0 = No Diabetes, 1 = Diabetes) |
| **Task** | Binary Classification |

**Why this dataset?** The Pima Indians Diabetes dataset is a classic benchmark in medical machine learning. It contains real clinical measurements from 768 female patients aged ≥21. The binary target (diabetic vs non-diabetic) is well-suited for classification, and known data-quality issues (impossible zeros) make it ideal for practising preprocessing pipelines. Predicting diabetes early has direct clinical impact — missing a positive case can lead to delayed treatment and serious complications.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_curve, roc_auc_score
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SCREENSHOTS_DIR = 'screenshots'
os.makedirs(SCREENSHOTS_DIR, exist_ok=True)

print('Libraries loaded successfully.')
print(f'NumPy: {np.__version__}  |  Pandas: {pd.__version__}')


Libraries loaded successfully.
NumPy: 1.26.2  |  Pandas: 2.1.3


---
# Part 1 — Data Preparation


In [2]:
df = pd.read_csv('diabetes.csv')
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
print('\nFirst 5 rows:')
df.head()


Dataset shape: 768 rows x 9 columns

First 5 rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
print('Data types and non-null counts:')
df.info()
print('\nBasic statistics:')
df.describe().round(2)


Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Basic statistics:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00,768.00
mean,3.85,120.89,69.11,20.54,79.80,31.99,0.47,33.24,0.35
std,3.37,31.97,19.36,15.95,115.24,7.88,0.33,11.76,0.48
min,0.00,0.00,0.00,0.00,0.00,0.00,0.08,21.00,0.00
25%,1.00,99.00,62.00,0.00,0.00,27.30,0.24,24.00,0.00
50%,3.00,117.00,72.00,23.00,30.50,32.00,0.37,29.00,0.00
75%,6.00,140.25,80.00,32.00,127.25,36.60,0.63,41.00,1.00
max,17.00,199.00,122.00,99.00,846.00,67.10,2.42,81.00,1.00


## 1.1 Handle Missing / Impossible Values

The dataset has no explicit NaN values, but several columns contain **biologically impossible zeros** (e.g., zero glucose, zero BMI). These are clearly measurement placeholders rather than true zero values. We replace them with the **median** of each column computed on non-zero rows.

**Why median?** Medians are robust to the skewed distributions present in clinical data and are unaffected by extreme outliers, unlike the mean.


In [4]:
zero_not_ok = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('Zero counts before cleaning:')
for col in zero_not_ok:
    n = (df[col] == 0).sum()
    pct = 100 * n / len(df)
    print(f'  {col:<20}: {n:>3} zeros ({pct:.1f}%)')

df_clean = df.copy()
for col in zero_not_ok:
    med = df_clean.loc[df_clean[col] != 0, col].median()
    df_clean[col] = df_clean[col].replace(0, med)

print('\nZero counts after cleaning:')
for col in zero_not_ok:
    print(f'  {col:<20}: {(df_clean[col] == 0).sum():>3} zeros')

print('\nMissing values after cleaning:', df_clean.isnull().sum().sum())


Zero counts before cleaning:
  Glucose             :   5 zeros (0.7%)
  BloodPressure       :  35 zeros (4.6%)
  SkinThickness       : 227 zeros (29.6%)
  Insulin             : 374 zeros (48.7%)
  BMI                 :  11 zeros (1.4%)

Zero counts after cleaning:
  Glucose             :   0 zeros
  BloodPressure       :   0 zeros
  SkinThickness       :   0 zeros
  Insulin             :   0 zeros
  BMI                 :   0 zeros

Missing values after cleaning: 0


## 1.2 Feature / Target Split

We separate the **8 clinical features** from the binary **Outcome** target.


In [5]:
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']
print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'\nClass distribution:\n{y.value_counts()}')
print(f'\nPositive class (diabetic): {y.mean()*100:.1f}%')


Features shape : (768, 8)
Target shape   : (768,)

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Positive class (diabetic): 34.9%


## 1.3 Train / Test Split (80 / 20)

We use `train_test_split` with `stratify=y` to preserve the class ratio in both splits and `random_state=42` for reproducibility.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set     : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nTrain class balance: {y_train.mean()*100:.1f}% positive')
print(f'Test  class balance: {y_test.mean()*100:.1f}% positive')


Training set : 614 samples (80%)
Test set     : 154 samples (20%)

Train class balance: 34.9% positive
Test  class balance: 35.1% positive


## 1.4 Feature Scaling — Standard Scaler

We apply **StandardScaler** (zero mean, unit variance): $x_{\text{scaled}} = \dfrac{x - \mu}{\sigma}$

**Why StandardScaler?**
- Logistic Regression and SVM are distance/gradient-based and highly sensitive to feature magnitude.
- Clinical features span very different ranges (e.g., Age 21–81 vs Insulin 0–846). Without scaling, high-magnitude features dominate the optimisation.
- StandardScaler handles outliers better than Min-Max scaling for skewed clinical data.

> **Important:** The scaler is **fit on the training set only**, then applied to both train and test sets to prevent data leakage.


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)        # transform only

print('Feature means after scaling (should be ~0):')
print(pd.Series(X_train_scaled.mean(axis=0), index=X.columns).round(4))
print('\nFeature std after scaling (should be ~1):')
print(pd.Series(X_train_scaled.std(axis=0), index=X.columns).round(4))


Feature means after scaling (should be ~0):
Pregnancies                -0.0
Glucose                    -0.0
BloodPressure               0.0
SkinThickness              -0.0
Insulin                    -0.0
BMI                        -0.0
DiabetesPedigreeFunction   -0.0
Age                        -0.0
dtype: float64

Feature std after scaling (should be ~1):
Pregnancies                 1.0
Glucose                     1.0
BloodPressure               1.0
SkinThickness               1.0
Insulin                     1.0
BMI                         1.0
DiabetesPedigreeFunction    1.0
Age                         1.0
dtype: float64


---
# Part 2 — Model Training


## Model 1: Logistic Regression

### How it works

Logistic Regression is a linear classification algorithm that models the **probability** that an input belongs to a class. It applies a linear combination of features $z = \mathbf{w}^T\mathbf{x} + b$ through the **sigmoid (logistic) function**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

The sigmoid squashes any real-valued number into $(0, 1)$, which we interpret as a probability. If $\sigma(z) \geq 0.5$ the model predicts class 1 (diabetic), otherwise class 0. Unlike linear regression, which can predict values outside $[0,1]$, the sigmoid always yields a valid probability. The model is trained by minimising **binary cross-entropy** loss, which heavily penalises confident wrong predictions. This makes logistic regression naturally suited for binary medical outcomes.


In [8]:
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr.fit(X_train_scaled, y_train)

coef_series = pd.Series(lr.coef_[0], index=X.columns).sort_values()
print('Logistic Regression coefficients (sorted):')
print(coef_series.round(4).to_string())
print()
print(f'Strongest positive influence : {coef_series.idxmax()} ({coef_series.max():.4f})')
print(f'Strongest negative influence : {coef_series.idxmin()} ({coef_series.min():.4f})')
print()
print(f'Interpretation:')
print(f'  A higher {coef_series.idxmax()} strongly INCREASES predicted diabetes probability.')
print(f'  A higher {coef_series.idxmin()} is associated with LOWER diabetes probability.')


Logistic Regression coefficients (sorted):
Insulin                    -0.0663
BloodPressure              -0.0444
SkinThickness               0.0279
Age                         0.1483
DiabetesPedigreeFunction    0.2333
Pregnancies                 0.3772
BMI                         0.6891
Glucose                     1.1826

Strongest positive influence : Glucose (1.1826)
Strongest negative influence : Insulin (-0.0663)

Interpretation:
  A higher Glucose strongly INCREASES predicted diabetes probability.
  A higher Insulin is associated with LOWER diabetes probability.


## Model 2: Support Vector Machine (SVM)

### How it works

An SVM finds the **optimal hyperplane** that separates the two classes with the **maximum margin**. The hyperplane is $\mathbf{w}^T\mathbf{x} + b = 0$. The **margin** is the perpendicular distance between the hyperplane and the nearest training points from each class — these critical points are called **support vectors**. Maximising the margin promotes better generalisation by keeping the decision boundary as far as possible from both classes.

The **C parameter** controls the trade-off between margin width and training errors:
- **Small C** → wide margin, more misclassifications tolerated (better on noisy data)
- **Large C** → narrow margin, fewer training errors (risk of overfitting)

For noisy medical data, a smaller C can be beneficial because perfect training separation may simply capture measurement noise rather than true clinical patterns.


In [9]:
for c_val in [0.01, 1.0]:
    svm_temp = SVC(kernel='linear', C=c_val, random_state=RANDOM_STATE, probability=True)
    svm_temp.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, svm_temp.predict(X_train_scaled))
    print(f'SVM C={c_val:<5} | Training accuracy: {train_acc*100:.2f}%')

print()
print('Choice: C=1.0')
print('  C=1.0 achieves higher training accuracy while C=0.01 under-fits the data.')
print('  For this dataset with 768 well-preprocessed samples, C=1.0 provides a good')
print('  bias-variance balance without overfitting.')

svm = SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE, probability=True)
svm.fit(X_train_scaled, y_train)
print('\nFinal SVM model trained (C=1.0, linear kernel).')


SVM C=0.01  | Training accuracy: 78.34%
SVM C=1.0   | Training accuracy: 78.66%

Choice: C=1.0
  C=1.0 achieves higher training accuracy while C=0.01 under-fits the data.
  For this dataset with 768 well-preprocessed samples, C=1.0 provides a good
  bias-variance balance without overfitting.

Final SVM model trained (C=1.0, linear kernel).


## Model 3: K-Nearest Neighbours (KNN)

### How it works

KNN is a **non-parametric, instance-based** algorithm. It stores all training samples and, at prediction time, finds the $k$ nearest training points to the query (using Euclidean distance by default). The predicted class is the **majority vote** among these $k$ neighbours. KNN has no explicit training phase — it is a lazy learner. It is sensitive to feature scaling (already applied) and to the choice of $k$: small $k$ gives high variance, large $k$ gives high bias. The default $k=5$ is a standard starting point that balances these effects and is widely used in clinical classification studies.


In [10]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

train_acc_knn = accuracy_score(y_train, knn.predict(X_train_scaled))
print(f'KNN (k=5) training accuracy: {train_acc_knn*100:.2f}%')
print('KNN model trained successfully.')


KNN (k=5) training accuracy: 83.06%
KNN model trained successfully.


---
# Part 3 — Model Evaluation

All models are evaluated on the **held-out test set** only (never the training set).


In [11]:
models = {
    'Logistic Regression': lr,
    'SVM (Linear, C=1)': svm,
    'KNN (k=5)': knn
}

def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'Accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'Precision':   round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':      round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score':    round(f1_score(y_true, y_pred, zero_division=0), 4),
        'Specificity': round(specificity, 4)
    }

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    results[name] = compute_metrics(y_test, y_pred)

print('Metrics computed for all models on test set.')


Metrics computed for all models on test set.


## 3.1 Confusion Matrices


In [12]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Pred: No', 'Pred: Yes'],
        yticklabels=['True: No', 'True: Yes']
    )
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices — Test Set', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOTS_DIR, '01_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/01_confusion_matrices.png')


Saved: screenshots/01_confusion_matrices.png


## 3.2 Performance Comparison Table


In [13]:
metrics_df = pd.DataFrame(results).T
metrics_df.index.name = 'Model'
print('=' * 75)
print('MODEL PERFORMANCE COMPARISON — TEST SET')
print('=' * 75)
print(metrics_df.to_string())
print('=' * 75)

print('\nBest per metric:')
for col in metrics_df.columns:
    best_model = metrics_df[col].idxmax()
    best_val   = metrics_df[col].max()
    print(f'  {col:<12}: {best_model} ({best_val:.4f})')


MODEL PERFORMANCE COMPARISON — TEST SET
                     Accuracy  Precision  Recall  F1-Score  Specificity
Model                                                                  
Logistic Regression    0.7078     0.6000  0.5000    0.5455         0.82
SVM (Linear, C=1)      0.7013     0.5909  0.4815    0.5306         0.82
KNN (k=5)              0.7532     0.6600  0.6111    0.6346         0.83

Best per metric:
  Accuracy    : KNN (k=5) (0.7532)
  Precision   : KNN (k=5) (0.6600)
  Recall      : KNN (k=5) (0.6111)
  F1-Score    : KNN (k=5) (0.6346)
  Specificity : KNN (k=5) (0.8300)


## 3.3 ROC Curves


In [14]:
plt.figure(figsize=(8, 6))

colors = ['steelblue', 'darkorange', 'seagreen']

for (name, model), color in zip(models.items(), colors):
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    results[name]['AUC'] = round(auc, 4)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.500)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves — All Models (Test Set)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOTS_DIR, '02_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/02_roc_curves.png')
print()
print('AUC Scores:')
for name in models:
    print(f'  {name:<25}: AUC = {results[name]["AUC"]:.4f}')


Saved: screenshots/02_roc_curves.png

AUC Scores:
  Logistic Regression      : AUC = 0.8130
  SVM (Linear, C=1)        : AUC = 0.8131
  KNN (k=5)                : AUC = 0.7886


## 3.4 Best Model & Clinical Discussion

After evaluating all three models on the test set, **KNN (k=5)** is the best overall model based on:

1. **Highest Recall (0.6111)** — In diabetes screening, missing a true positive (a diabetic patient classified as healthy) is far more dangerous than a false alarm. KNN correctly identifies the most diabetic patients among all three models.
2. **Highest F1-Score (0.6346)** — F1 balances precision and recall; KNN's F1 of 0.635 vs 0.546 (LR) and 0.531 (SVM) demonstrates superior overall classification quality on this imbalanced dataset.
3. **Highest Accuracy (75.3%)** and **Highest Specificity (0.83)** — KNN is the strongest performer across all point-estimate metrics.

> **Note on AUC:** Logistic Regression (0.813) and SVM (0.813) achieve a marginally higher AUC than KNN (0.789), meaning they rank positive cases slightly better across *all* thresholds. However, at the default 0.5 threshold KNN is clearly superior in every metric that matters for clinical screening.

> **Clinical Note:** For diabetes screening, **Recall is more important than Accuracy**. A patient missed by the model receives no treatment or follow-up, potentially leading to serious complications (neuropathy, retinopathy, cardiovascular disease). A false positive leads only to additional testing — a far less harmful outcome. KNN's recall of 61% vs LR's 50% means it catches ~22% more diabetic patients, a clinically significant improvement.


---
# Part 4 — Cross-Validation

We apply **5-fold stratified cross-validation** to the best model from Part 3: **KNN (k=5)**.

**Why cross-validation?** A single 80/20 split gives one estimate that depends on which samples
happen to land in each split. With only 768 rows this variance can be large. K-fold CV partitions the
data into $k$ equal folds, trains on $k{-}1$ folds and validates on 1, repeating $k$ times so every
sample is used for validation exactly once. The result is a mean accuracy with a standard deviation
that quantifies model stability.


In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_all = df_clean.drop(columns=['Outcome']).values
y_all = df_clean['Outcome'].values

cv_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

fold_scores = cross_val_score(cv_pipeline, X_all, y_all, cv=cv, scoring='accuracy')

print('5-Fold Cross-Validation Results (KNN k=5):')
print('-' * 50)
for i, score in enumerate(fold_scores, 1):
    print(f'  Fold {i}: {score*100:.2f}%')
print('-' * 50)
print(f'  Mean Accuracy : {fold_scores.mean()*100:.2f}%')
print(f'  Std Deviation : {fold_scores.std()*100:.2f}%')
print()
test_acc_best = results['KNN (k=5)']['Accuracy']
cv_mean       = fold_scores.mean()
print(f'Single test-set accuracy  : {test_acc_best*100:.2f}%')
print(f'5-Fold CV mean accuracy   : {cv_mean*100:.2f}%')
print(f'Difference                : {abs(test_acc_best - cv_mean)*100:.2f} percentage points')


5-Fold Cross-Validation Results (KNN k=5):
--------------------------------------------------
  Fold 1: 73.38%
  Fold 2: 74.68%
  Fold 3: 74.68%
  Fold 4: 72.55%
  Fold 5: 66.67%
--------------------------------------------------
  Mean Accuracy : 72.39%
  Std Deviation : 2.97%

Single test-set accuracy  : 75.32%
5-Fold CV mean accuracy   : 72.39%
Difference                : 2.93 percentage points


In [19]:
fig, ax = plt.subplots(figsize=(8, 4))

fold_labels = [f'Fold {i}' for i in range(1, 6)]
bars = ax.bar(fold_labels, fold_scores * 100, color='seagreen', alpha=0.8, edgecolor='black')
ax.axhline(fold_scores.mean() * 100, color='red', linestyle='--', linewidth=2,
           label=f'CV Mean = {fold_scores.mean()*100:.2f}%')
ax.axhline(test_acc_best * 100, color='darkorange', linestyle=':', linewidth=2,
           label=f'Single Test-Set = {test_acc_best*100:.2f}%')

for bar, score in zip(bars, fold_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{score*100:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_ylim(60, 90)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('5-Fold Cross-Validation — KNN (k=5)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOTS_DIR, '03_cross_validation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: screenshots/03_cross_validation.png')


Saved: screenshots/03_cross_validation.png


## Cross-Validation Discussion

**Comparison:** The 5-fold CV mean accuracy (72.39%) is ~3 percentage points below the single test-set accuracy (75.32%). This modest gap is expected: the single split happened to produce a slightly favourable test set, while CV averages over 5 different partitions to give a more realistic estimate of ~72% generalisation accuracy.

**Why CV gives a more reliable estimate:**
1. **Reduced variance:** A single split can be lucky or unlucky depending on which rows fall in the test set. CV averages over 5 different test subsets, producing a far more stable estimate.
2. **Full data utilisation:** With only 768 samples, holding out 20% (≈154 rows) wastes information. CV ensures every sample is used for validation exactly once, making full use of the limited data.
3. **Stability signal:** The standard deviation of 2.97% across folds is small but non-trivial, reflecting the natural variation in a moderately-sized clinical dataset. This tells us the model is reasonably stable but could benefit from more training data — a common challenge in medical settings where data collection is expensive.


In [20]:
print('=' * 70)
print('FINAL SUMMARY')
print('=' * 70)
summary_df = pd.DataFrame(results).T
summary_df.index.name = 'Model'
print(summary_df.to_string())
print('=' * 70)
print()
print('Best model: KNN (k=5)')
print('  - Highest Accuracy  : 75.3%  (vs LR 70.8%, SVM 70.1%)')
print('  - Highest Recall    : 0.611  (vs LR 0.500, SVM 0.481)  ← most clinically important')
print('  - Highest F1-Score  : 0.635  (vs LR 0.546, SVM 0.531)')
print('  - Highest Precision : 0.660  (vs LR 0.600, SVM 0.591)')
print('  - Highest Specificity: 0.830 (vs LR 0.820, SVM 0.820)')
print()
print('Note: LR and SVM have marginally higher AUC (0.813 vs KNN 0.789),')
print('      but KNN is superior at the standard 0.5 threshold on all metrics.')


FINAL SUMMARY
                     Accuracy  Precision  Recall  F1-Score  Specificity     AUC
Model                                                                          
Logistic Regression    0.7078     0.6000  0.5000    0.5455         0.82  0.8130
SVM (Linear, C=1)      0.7013     0.5909  0.4815    0.5306         0.82  0.8131
KNN (k=5)              0.7532     0.6600  0.6111    0.6346         0.83  0.7886

Best model: KNN (k=5)
  - Highest Accuracy  : 75.3%  (vs LR 70.8%, SVM 70.1%)
  - Highest Recall    : 0.611  (vs LR 0.500, SVM 0.481)  ← most clinically important
  - Highest F1-Score  : 0.635  (vs LR 0.546, SVM 0.531)
  - Highest Precision : 0.660  (vs LR 0.600, SVM 0.591)
  - Highest Specificity: 0.830 (vs LR 0.820, SVM 0.820)

Note: LR and SVM have marginally higher AUC (0.813 vs KNN 0.789),
      but KNN is superior at the standard 0.5 threshold on all metrics.


---
# Conclusion

This notebook applied the full supervised machine learning pipeline to the Pima Indians Diabetes dataset:

| Step | Action |
|:-----|:-------|
| Data Cleaning | Replaced impossible zeros with column medians |
| Splitting | Stratified 80/20 split (random_state=42) |
| Scaling | StandardScaler fit on training set only |
| Model 1 | Logistic Regression — linear, probabilistic, interpretable |
| Model 2 | SVM (linear, C=1) — maximum-margin hyperplane |
| Model 3 | KNN (k=5) — instance-based, non-parametric |
| Evaluation | Confusion matrix, Accuracy, Precision, Recall, F1, Specificity, AUC |
| CV | 5-fold stratified cross-validation on best model |

**KNN (k=5)** is selected as the best model. It achieves the highest scores on every point-estimate metric — including **Recall (0.611)**, which is the most clinically important metric for diabetes screening. KNN correctly identifies 61% of all true diabetic patients compared to 50% for Logistic Regression and 48% for SVM.

While Logistic Regression has a slightly higher AUC (0.813), its lower Recall at the default threshold makes it less appropriate for a screening tool where **missing a diagnosis is worse than a false alarm**.
